# Population by Elevation Companion Analysis

This companion notebook supports the repository-level hypsographic demography summary and animation. It is not a core manuscript figure notebook. It uses the global integer-elevation age-sex population tables to summarize population below selected elevation thresholds and to animate the 2025 population pyramid as the elevation threshold descends.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "data" / "dataset_s1_hypsographic_demography.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "data" / "dataset_s1_hypsographic_demography.csv").exists():
    ROOT = Path("/Users/f6z/Documents/projects/montana_state/hypsographic_demography")

DATA = ROOT / "data"
FIG = ROOT / "outputs" / "figures"
TAB = ROOT / "outputs" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

INPUT_GLOBAL_ELEVATION_AGE_SEX = DATA / "global_integer_elevation_age_sex_2015_2025.parquet"
YEARS = [2015, 2025]
OUT_TABLE = TAB / "population_by_elevation_threshold_summary.csv"
OUT_AGE_TABLE = TAB / "population_by_elevation_threshold_age_summary.csv"
OUT_GIF = FIG / "population_by_elevation_pyramid.gif"

YEAR_COL = "year"
GEOGRAPHY_COL = "geography_level"
CONTINENT_COL = "continent"
ELEV_COL = "elevation_m"
SEX_COL = "sex"
AGE_LABEL_COL = "age_label"
BROAD_AGE_COL = "broad_age_group"
POP_COL = "population_count"

BROAD_AGE_LABELS = {
    "young_0_14": "Young (0-14)",
    "working_age_15_64": "Working age (15-64)",
    "old_age_65_plus": "Older (65+)",
    "older_65_plus": "Older (65+)",
}
BROAD_AGE_ORDER = ["young_0_14", "working_age_15_64", "old_age_65_plus", "older_65_plus"]

THRESHOLDS_M = [100, 150, 200, 500, 1000, 1500, 3500]
MIN_ELEV_M = 0
MAX_ELEV_M = 3500
FRAME_COUNT = 180
FPS = 12
DPI = 120

MALE_COLOR = "#2D004B"
FEMALE_COLOR = "#F05A28"
DOT_COLOR = "#2A9D8F"
GRID_COLOR = "#D9D9D9"
AXIS_COLOR = "#111111"
TEXT_COLOR = "#111111"
LINE_COLOR = "#353535"
SHADE_COLOR = "#9E9E9E"

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def load_global_year(year: int) -> pd.DataFrame:
    path = INPUT_GLOBAL_ELEVATION_AGE_SEX
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path.relative_to(ROOT)}")
    df = pd.read_parquet(path)
    if YEAR_COL in df.columns:
        df = df[df[YEAR_COL].astype(int).eq(year)].copy()
    df[ELEV_COL] = pd.to_numeric(df[ELEV_COL], errors="coerce")
    df[POP_COL] = pd.to_numeric(df[POP_COL], errors="coerce").fillna(0)
    df = df.dropna(subset=[ELEV_COL])
    if df.empty:
        raise ValueError(f"No global integer-elevation rows found for {year}.")
    return df


def make_5yr_age_bin(age_label) -> str:
    s = str(age_label).strip().lower()
    nums = re.findall(r"\d+", s)
    if not nums:
        return s
    lo = int(nums[0])
    if lo < 5:
        return "0-4"
    if "+" in s:
        return f"{(lo // 5) * 5}+"
    lo5 = (lo // 5) * 5
    return f"{lo5}-{lo5 + 4}"


def age_bin_sort_value(age_bin) -> int:
    nums = re.findall(r"\d+", str(age_bin))
    return int(nums[0]) if nums else 999


def clean_sex(x) -> str:
    s = str(x).strip().lower()
    if s in {"m", "male", "men", "1"}:
        return "Male"
    if s in {"f", "female", "women", "2"}:
        return "Female"
    return str(x)


def fmt_population(x, pos=None) -> str:
    x = abs(x)
    if x >= 1e9:
        return f"{x / 1e9:.1f}B"
    if x >= 1e6:
        return f"{x / 1e6:.0f}M"
    return f"{x:,.0f}"


def tidy_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(AXIS_COLOR)
        spine.set_linewidth(0.8)
    ax.tick_params(axis="both", colors=AXIS_COLOR, width=0.8, length=3.5, direction="out")

print("Repo root:", ROOT)


## Threshold Summary

The threshold table includes 150 m because the headline companion result is that roughly half of the global population lives at or below 150 m elevation.

In [ ]:
threshold_rows = []
threshold_age_rows = []
loaded = {}

for year in YEARS:
    df = load_global_year(year)
    loaded[year] = df
    total_pop = df[POP_COL].sum()
    print(f"{year}: {len(df):,} global rows; total population {total_pop:,.0f}")

    for threshold in THRESHOLDS_M:
        pop_below = df.loc[df[ELEV_COL] <= threshold, POP_COL].sum()
        pop_above = total_pop - pop_below
        threshold_rows.append({
            "year": year,
            "threshold_m": threshold,
            "population_below_or_equal": pop_below,
            "percent_below_or_equal": pop_below / total_pop * 100,
            "population_above": pop_above,
            "percent_above": pop_above / total_pop * 100,
        })

    age_totals = df.groupby(BROAD_AGE_COL, as_index=False)[POP_COL].sum()
    for age_row in age_totals.itertuples(index=False):
        broad_age_group = getattr(age_row, BROAD_AGE_COL)
        age_total = getattr(age_row, POP_COL)
        age_df = df[df[BROAD_AGE_COL].eq(broad_age_group)]

        for threshold in THRESHOLDS_M:
            pop_below = age_df.loc[age_df[ELEV_COL] <= threshold, POP_COL].sum()
            pop_above = age_total - pop_below
            threshold_age_rows.append({
                "year": year,
                "threshold_m": threshold,
                "broad_age_group": broad_age_group,
                "age_group_label": BROAD_AGE_LABELS.get(broad_age_group, broad_age_group),
                "population_below_or_equal": pop_below,
                "percent_below_or_equal_within_age_group": pop_below / age_total * 100,
                "population_above": pop_above,
                "percent_above_within_age_group": pop_above / age_total * 100,
                "age_group_total_population": age_total,
            })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv(OUT_TABLE, index=False)

threshold_age_summary = pd.DataFrame(threshold_age_rows)
threshold_age_summary["broad_age_sort"] = threshold_age_summary["broad_age_group"].map(
    {age: i for i, age in enumerate(BROAD_AGE_ORDER)}
).fillna(len(BROAD_AGE_ORDER))
threshold_age_summary = (
    threshold_age_summary
    .sort_values(["year", "threshold_m", "broad_age_sort", "broad_age_group"])
    .drop(columns="broad_age_sort")
)
threshold_age_summary.to_csv(OUT_AGE_TABLE, index=False)

below_150 = threshold_summary[threshold_summary["threshold_m"].eq(150)].copy()
below_150_by_age = threshold_age_summary[threshold_age_summary["threshold_m"].eq(150)].copy()
print(f"Saved {OUT_TABLE.relative_to(ROOT)}")
print(f"Saved {OUT_AGE_TABLE.relative_to(ROOT)}")

display(
    threshold_summary.style.format({
        "threshold_m": "{:,.0f}",
        "population_below_or_equal": "{:,.0f}",
        "percent_below_or_equal": "{:.2f}%",
        "population_above": "{:,.0f}",
        "percent_above": "{:.2f}%",
    })
)

display(
    below_150_by_age.style.format({
        "threshold_m": "{:,.0f}",
        "population_below_or_equal": "{:,.0f}",
        "percent_below_or_equal_within_age_group": "{:.2f}%",
        "population_above": "{:,.0f}",
        "percent_above_within_age_group": "{:.2f}%",
        "age_group_total_population": "{:,.0f}",
    })
)

display(Markdown(
    "**Key result:** in 2025, "
    f"{below_150.loc[below_150['year'].eq(2025), 'percent_below_or_equal'].iloc[0]:.1f}% "
    "of the global population lived at or below 150 m elevation "
    f"({below_150.loc[below_150['year'].eq(2015), 'percent_below_or_equal'].iloc[0]:.1f}% in 2015). "
    "By broad age group in 2025, the corresponding shares were "
    + ", ".join(
        f"{row.age_group_label}: {row.percent_below_or_equal_within_age_group:.1f}%"
        for row in below_150_by_age[below_150_by_age["year"].eq(2025)].itertuples(index=False)
    )
    + "."
))


## Population-By-Elevation GIF

The animation shows the 2025 cumulative population below each descending elevation threshold, split by age and sex.

In [ ]:
df = loaded.get(2025)
if df is None:
    df = load_global_year(2025)

df = df.copy()
df["sex_clean"] = df[SEX_COL].apply(clean_sex)
df = df[df["sex_clean"].isin(["Male", "Female"])].copy()
df["age_bin"] = df[AGE_LABEL_COL].apply(make_5yr_age_bin)
df["elevation_capped_m"] = df[ELEV_COL].clip(lower=MIN_ELEV_M, upper=MAX_ELEV_M).round().astype(int)

g = (
    df.groupby(["elevation_capped_m", "age_bin", "sex_clean"], as_index=False)[POP_COL]
      .sum()
      .rename(columns={"elevation_capped_m": "elevation_m", POP_COL: "population"})
)

age_order = sorted(g["age_bin"].unique(), key=age_bin_sort_value)
sex_order = ["Male", "Female"]
elevations = np.arange(MIN_ELEV_M, MAX_ELEV_M + 1)

full_index = pd.MultiIndex.from_product(
    [elevations, age_order, sex_order],
    names=["elevation_m", "age_bin", "sex_clean"],
)

wide = (
    g.set_index(["elevation_m", "age_bin", "sex_clean"])["population"]
     .reindex(full_index, fill_value=0)
     .reset_index()
)
wide["cum_below"] = (
    wide.sort_values("elevation_m")
        .groupby(["age_bin", "sex_clean"])["population"]
        .cumsum()
)

cum = (
    wide.pivot_table(
        index="elevation_m",
        columns=["age_bin", "sex_clean"],
        values="cum_below",
        fill_value=0,
    )
    .sort_index()
)

cdf = wide.groupby("elevation_m", as_index=False)["population"].sum().sort_values("elevation_m")
cdf["cum_below"] = cdf["population"].cumsum()
cdf["cum_frac"] = cdf["cum_below"] / cdf["cum_below"].iloc[-1]

full_ref = cum.loc[MAX_ELEV_M]
full_male = np.array([full_ref.get((age, "Male"), 0) for age in age_order], dtype=float)
full_female = np.array([full_ref.get((age, "Female"), 0) for age in age_order], dtype=float)
xmax = max(full_male.max(), full_female.max()) * 1.12
ypos = np.arange(len(age_order))

thresholds = np.linspace(MAX_ELEV_M, MIN_ELEV_M, FRAME_COUNT)
thresholds = pd.unique(np.round(thresholds).astype(int))
print(f"Animation frames: {len(thresholds):,}; duration: {len(thresholds) / FPS:.1f} s")


def get_pyramid_values(threshold: int):
    row = cum.loc[int(threshold)]
    male = np.array([row.get((age, "Male"), 0) for age in age_order], dtype=float)
    female = np.array([row.get((age, "Female"), 0) for age in age_order], dtype=float)
    return male, female

fig = plt.figure(figsize=(9.0, 6.2))
gs = GridSpec(1, 2, width_ratios=[3.25, 1.45], wspace=0.25, figure=fig)
ax_pyr = fig.add_subplot(gs[0, 0])
ax_cdf = fig.add_subplot(gs[0, 1])
fig.subplots_adjust(top=0.88, left=0.085, right=0.965, bottom=0.13)
fig.suptitle("Hypsographic demography: population by elevation (2025)", y=0.955, fontsize=13.5, color=TEXT_COLOR)


def draw_frame(i):
    threshold = int(thresholds[i])
    male, female = get_pyramid_values(threshold)
    cdf_row = cdf.loc[cdf["elevation_m"].eq(threshold)]
    cum_pop = float(cdf_row["cum_below"].iloc[0])
    cum_frac = float(cdf_row["cum_frac"].iloc[0]) * 100.0

    ax_pyr.clear()
    ax_cdf.clear()
    ax_pyr.set_axisbelow(True)
    ax_cdf.set_axisbelow(True)

    ax_pyr.grid(True, which="major", axis="x", linestyle="--", linewidth=0.55, color=GRID_COLOR, alpha=0.85, zorder=0)
    ax_pyr.barh(ypos, -full_male, height=0.84, color=MALE_COLOR, alpha=0.13, edgecolor="none", zorder=2)
    ax_pyr.barh(ypos, full_female, height=0.84, color=FEMALE_COLOR, alpha=0.13, edgecolor="none", zorder=2)
    ax_pyr.barh(ypos, -male, height=0.60, color=MALE_COLOR, alpha=0.96, edgecolor=AXIS_COLOR, linewidth=0.25, zorder=3)
    ax_pyr.barh(ypos, female, height=0.60, color=FEMALE_COLOR, alpha=0.96, edgecolor=AXIS_COLOR, linewidth=0.25, zorder=3)
    ax_pyr.axvline(0, color=AXIS_COLOR, linewidth=0.8, zorder=4)
    ax_pyr.set_yticks(ypos)
    ax_pyr.set_yticklabels(age_order)
    ax_pyr.set_xlim(-xmax, xmax)
    ax_pyr.xaxis.set_major_formatter(FuncFormatter(fmt_population))
    ax_pyr.set_xlabel("Population")
    ax_pyr.set_ylabel("Age group")
    ax_pyr.text(
        0.02,
        0.985,
        f"Below {threshold:,.0f} m: {cum_pop / 1e9:.2f}B ({cum_frac:.1f}%)",
        transform=ax_pyr.transAxes,
        ha="left",
        va="top",
        fontsize=9.5,
        color=TEXT_COLOR,
    )
    ax_pyr.text(0.25, -0.085, "Male", transform=ax_pyr.transAxes, ha="center", va="top", fontsize=9.5, color=TEXT_COLOR)
    ax_pyr.text(0.75, -0.085, "Female", transform=ax_pyr.transAxes, ha="center", va="top", fontsize=9.5, color=TEXT_COLOR)
    tidy_axes(ax_pyr)

    ax_cdf.grid(True, which="major", linestyle="--", linewidth=0.55, color=GRID_COLOR, alpha=0.85)
    ax_cdf.fill_betweenx(cdf["elevation_m"], 0, cdf["cum_frac"] * 100, color=SHADE_COLOR, alpha=0.16, zorder=1)
    ax_cdf.plot(cdf["cum_frac"] * 100, cdf["elevation_m"], color=LINE_COLOR, linewidth=1.8, alpha=0.98, zorder=3)
    ax_cdf.scatter([cum_frac], [threshold], s=55, color=DOT_COLOR, edgecolor=AXIS_COLOR, linewidth=0.9, zorder=4)
    ax_cdf.axhline(threshold, color=AXIS_COLOR, linewidth=0.8, alpha=0.45, linestyle="--", zorder=2)
    ax_cdf.set_xlim(0, 100)
    ax_cdf.set_ylim(MIN_ELEV_M, MAX_ELEV_M)
    ax_cdf.set_xlabel("Cumulative population\nbelow threshold (%)")
    ax_cdf.set_ylabel("Elevation threshold (m)")
    ax_cdf.text(0.04, 0.985, f"{threshold:,.0f} m", transform=ax_cdf.transAxes, ha="left", va="top", fontsize=9.5, color=TEXT_COLOR)
    tidy_axes(ax_cdf)
    return []

anim = FuncAnimation(fig, draw_frame, frames=len(thresholds), interval=1000 / FPS, blit=False)
anim.save(OUT_GIF, writer=PillowWriter(fps=FPS), dpi=DPI)
plt.close(fig)

print(f"Saved {OUT_GIF.relative_to(ROOT)}")
display(Markdown(f"Animation saved to `{OUT_GIF.relative_to(ROOT)}`."))
